In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Load datasets

In [3]:
books = pd.read_csv("../data/BX-Books.csv", sep=";", encoding="latin-1", on_bad_lines="skip")
users = pd.read_csv("../data/BX-Users.csv", sep=";", encoding="latin-1", on_bad_lines="skip")
ratings = pd.read_csv("../data/BX-Book-Ratings.csv", sep=";", encoding="latin-1", on_bad_lines="skip")

C:\Users\sathi\AppData\Local\Temp\ipykernel_11156\2376582546.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("../data/BX-Books.csv", sep=";", encoding="latin-1", on_bad_lines="skip")


In [4]:
print(books.info())
print(books.isnull().sum().sum())

<class 'pandas.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  str   
 1   Book-Title           271360 non-null  str   
 2   Book-Author          271358 non-null  str   
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  str   
 5   Image-URL-S          271360 non-null  str   
 6   Image-URL-M          271360 non-null  str   
 7   Image-URL-L          271357 non-null  str   
dtypes: object(1), str(7)
memory usage: 83.4+ MB
None
7


In [5]:
print(users.info())
print(users.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 278858 entries, 0 to 278857
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   User-ID   278858 non-null  int64  
 1   Location  278858 non-null  str    
 2   Age       168096 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 13.4 MB
None
User-ID          0
Location         0
Age         110762
dtype: int64


In [6]:
print(ratings.info())
print(ratings.isnull().sum().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   User-ID      1149780 non-null  int64
 1   ISBN         1149780 non-null  str  
 2   Book-Rating  1149780 non-null  int64
dtypes: int64(2), str(1)
memory usage: 37.3 MB
None
0


In [7]:
print(books.shape, users.shape, ratings.shape, sep='\n')

(271360, 8)
(278858, 3)
(1149780, 3)


### Data Cleaning

In [8]:
# Keep only important columns.
books = books[['ISBN','Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher','Image-URL-L']]

In [9]:
# Remove missing values.
books.dropna(inplace=True)

In [10]:
# Remove duplicate books.
books.drop_duplicates(inplace=True)

 ### Exploratory Data Analysis (EDA)

In [11]:
# Understand dataset.
ratings['User-ID'].value_counts().shape

(105283,)

In [12]:
ratings['User-ID'].unique().shape

(105283,)

In [13]:
# Filtering Users,who had at least rated more than 100 books
x = ratings['User-ID'].value_counts() > 100

In [14]:
x[x].shape

(1825,)

In [15]:
y= x[x].index

In [16]:
y

Index([ 11676, 198711, 153662,  98391,  35859, 212898, 278418,  76352, 110973,
       235105,
       ...
        99441, 104657, 140879, 175636, 187410, 189666, 205383, 238186, 262070,
       266283],
      dtype='int64', name='User-ID', length=1825)

In [17]:

ratings = ratings[ratings['User-ID'].isin(y)]
ratings.head()

,User-ID,ISBN,Book-Rating
412,276925,0006511929,0
413,276925,002542730X,10
414,276925,0060520507,0
415,276925,0060930934,0
416,276925,0060951303,0


In [18]:
ratings.shape

(656605, 3)

In [19]:
# Now join ratings with books

ratings_with_books = ratings.merge(books, on='ISBN')

In [20]:
ratings_with_books.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L
0,276925,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...
1,276925,0060520507,0,"Sushi for Beginners : A Novel (Keyes, Marian)",Marian Keyes,2003,William Morrow,http://images.amazon.com/images/P/0060520507.0...
2,276925,0060930934,0,Wasted : A Memoir of Anorexia and Bulimia,Marya Hornbacher,1999,Perennial,http://images.amazon.com/images/P/0060930934.0...
3,276925,0060951303,0,La casa de los espÃ­ritus,Isabel Allende,1995,Rayo,http://images.amazon.com/images/P/0060951303.0...
4,276925,0140154078,6,The Music of Chance,Paul Auster,1993,Penguin Books,http://images.amazon.com/images/P/0140154078.0...


In [21]:
ratings_with_books.shape

(604847, 8)

In [22]:
number_rating = ratings_with_books.groupby('Book-Title')['Book-Rating'].count().reset_index()

In [23]:
number_rating.head()

,Book-Title,Book-Rating
0,A Light in the Storm: The Civil War Diary of ...,3
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [24]:
number_rating.rename(columns={'Book-Rating':'num_of_rating'},inplace=True)

In [25]:
number_rating.head()

,Book-Title,num_of_rating
0,A Light in the Storm: The Civil War Diary of ...,3
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [26]:
final_rating = ratings_with_books.merge(number_rating, on='Book-Title')

In [27]:
final_rating.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L,num_of_rating
0,276925,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,105
1,276925,0060520507,0,"Sushi for Beginners : A Novel (Keyes, Marian)",Marian Keyes,2003,William Morrow,http://images.amazon.com/images/P/0060520507.0...,16
2,276925,0060930934,0,Wasted : A Memoir of Anorexia and Bulimia,Marya Hornbacher,1999,Perennial,http://images.amazon.com/images/P/0060930934.0...,12
3,276925,0060951303,0,La casa de los espÃ­ritus,Isabel Allende,1995,Rayo,http://images.amazon.com/images/P/0060951303.0...,8
4,276925,0140154078,6,The Music of Chance,Paul Auster,1993,Penguin Books,http://images.amazon.com/images/P/0140154078.0...,6


In [28]:
final_rating.shape

(604847, 9)

In [29]:
# Lets take those books which got at least 50 rating of user

final_rating = final_rating[final_rating['num_of_rating'] >= 50]

In [30]:
final_rating.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L,num_of_rating
0,276925,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,105
5,276925,0140327592,0,Matilda,Roald Dahl,1990,Viking Penguin Inc,http://images.amazon.com/images/P/0140327592.0...,79
12,276925,0316666343,0,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown",http://images.amazon.com/images/P/0316666343.0...,430
15,276925,0385504209,8,The Da Vinci Code,Dan Brown,2003,Doubleday,http://images.amazon.com/images/P/0385504209.0...,335
37,276925,0804106304,0,The Joy Luck Club,Amy Tan,1994,Prentice Hall (K-12),http://images.amazon.com/images/P/0804106304.0...,263


In [31]:
final_rating.shape

(100426, 9)

In [32]:
# lets drop the duplicates
final_rating.drop_duplicates(['User-ID','Book-Title'],inplace=True)

In [33]:
final_rating.shape

(97962, 9)

### Popularity Recommendation

In [34]:
popular_books = final_rating.groupby('Book-Title').count()['Book-Rating'].reset_index()
popular_books.rename(columns={'Book-Rating':'num_ratings'}, inplace=True)

popular_books = popular_books.sort_values('num_ratings', ascending=False)

popular_books.head(10)

,Book-Title,num_ratings
1077,Wild Animus,615
890,The Lovely Bones: A Novel,430
789,The Da Vinci Code,334
136,Bridget Jones's Diary,320
232,Divine Secrets of the Ya-Ya Sisterhood: A Novel,310
898,The Nanny Diaries: A Novel,306
908,The Pelican Brief,294
818,The Firm,291
30,A Painted House,289
949,The Secret Life of Bees,285


### Pivot Table Creation

In [35]:
# Lets create a pivot table
def create_pivot_table(data):
    
    pivot = data.pivot_table(
        index='Book-Title',
        columns='User-ID',
        values='Book-Rating'
    )
    
    pivot.fillna(0, inplace=True)
    
    return pivot

In [36]:
book_pivot = create_pivot_table(final_rating)
book_pivot

User-ID,183,254,507,882,1424,1435,1733,1903,2033,2110,...,276018,276463,276680,276925,277427,277478,277639,278137,278188,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2010: Odyssey Two,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
204 Rosewood Lane,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24 Hours,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
book_pivot.shape


(1094, 1794)

### Training Model (KNN Collaborative Filtering)


In [38]:
from scipy.sparse import csr_matrix

In [39]:
book_sparse = csr_matrix(book_pivot)

In [40]:
type(book_sparse)

scipy.sparse._csr.csr_matrix

In [41]:
def train_model(pivot_table):

    from sklearn.neighbors import NearestNeighbors
    from scipy.sparse import csr_matrix

    sparse_matrix = csr_matrix(pivot_table)

    model = NearestNeighbors(
        metric='cosine',
        algorithm='brute'
    )

    model.fit(sparse_matrix)

    return model

In [42]:
model = train_model(book_pivot)

### Model Evaluation

In [43]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity matrix
similarity_scores = cosine_similarity(book_pivot)

print("Similarity matrix shape:", similarity_scores.shape)

Similarity matrix shape: (1094, 1094)


In [44]:
# Simple Recommendation Test
query_index = 10

distances, indices = model.kneighbors(book_pivot.iloc[query_index, :].values.reshape(1, -1), n_neighbors=6)

print("Recommendations for:", book_pivot.index[query_index])

for i in range(1, len(indices.flatten())):
    print(book_pivot.index[indices.flatten()[i]])

Recommendations for: A Bend in the Road
A Walk to Remember
Toxin
The Last Time They Met : A Novel
The Rescue
Every Breath You Take : A True Story of Obsession, Revenge, and Murder


In [45]:
#keeping books name
book_names = book_pivot.index

### Recomdation fuction

In [51]:
# fetch_poster
def fetch_poster(suggestion):

    book_name = []
    poster_url = []

    for book_id in suggestion[0]:
        book_name.append(book_pivot.index[book_id])

    for name in book_name:
        url = books.loc[books['Book-Title'] == name, 'Image-URL-L'].values[0]
        poster_url.append(url)

    return book_name, poster_url

In [47]:
# recommend_books
def recommend_books(book_name):

    if book_name not in book_pivot.index:
        return ["Book not found in dataset"]

    index = np.where(book_pivot.index == book_name)[0][0]

    distances, suggestions = model.kneighbors(
        book_pivot.iloc[index, :].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []

    for i in suggestions[0][1:]:
        recommended_books.append(book_pivot.index[i])

    return recommended_books

### Random Recommendation Test

In [48]:
import random
random_book = random.choice(list(book_pivot.index))
print("Input Book:", random_book)
print("Recommended Books:")

for book in recommend_books(random_book):
    print(book)

Input Book: The Lost Boy: A Foster Child's Search for the Love of a Family
Recommended Books:
A Child Called \It\": One Child's Courage to Survive"
A Man Named Dave: A Story of Triumph and Forgiveness
Left Behind: A Novel of the Earth's Last Days (Left Behind #1)
The Long Road Home
The Ranch


In [49]:
import pickle
pickle.dump(model, open('../artifacts/model.pkl','wb'))
pickle.dump(book_pivot, open('../artifacts/book_pivot.pkl','wb'))
pickle.dump(final_rating, open('../artifacts/final_rating.pkl','wb'))
pickle.dump(book_names, open('../artifacts/book_names.pkl','wb'))